# Optional Example: AI Sentiment Pipeline Deep Dive

In this optional example, we open the news ingestion pipeline that backs the intraday engine. The Core notebooks treat the news scorer as a black box (the runner reads the latest hourly artifact and joins severity per trade); here we trace one headline end-to-end through that black box, recover the news loadings $\nu_i$ from a controlled experiment, and run a held-out test to see whether the loadings generalize. This is required reading if we plan to swap the synthetic corpus for a real news API on May 11 and want to know what the engine is actually doing with the LLM scores.

> __Learning Objectives:__
>
> * __Trace one headline through the pipeline:__ Generate a synthetic news corpus, inspect the [`MyNewsItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) fields, score it via [`score_news_with_claude!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#score_news_with_claude!), and observe how noise enters the cached-score path.
> * __Recover the planted news loadings:__ Build a controlled experiment with known $\nu_i$ values, run [`estimate_sim_with_news`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#estimate_sim_with_news), and quantify how well the estimator recovers the planted loadings.
> * __Diagnose overfitting on held-out data:__ Train on the first half of the sample, hold out the second half, and compare in-sample versus out-of-sample correlation between `news_factor` and realized residual growth.

Let's open the box.

___

## Setup, Data and Prerequisites

In [1]:
include("Include.jl");

### Constants


In [2]:
# Controlled-experiment knobs
T_DAYS = 600                          # total trading days simulated
K_TICKERS = 3                         # small basket so plots stay readable
TICKERS_DEMO = ["AAA", "BBB", "CCC"]
TRUE_ALPHA = [0.0001, 0.0002, 0.00005]
TRUE_BETA = [1.10, 0.80, 1.40]
TRUE_NU = [0.020, 0.005, 0.030]       # planted news loadings to recover
TRUE_SIGMA_EPS = [0.008, 0.006, 0.012]
MARKET_DRIFT = 0.0003
MARKET_VOL = 0.012
MASTER_SEED = 11_2026


112026

Build a controlled price-and-news fixture. The market index follows a calibrated geometric Brownian path; per-ticker prices follow a Single-Index Model contaminated by news shocks via the planted $\nu_i$ loadings. In the code block below, we return: `market::Vector{Float64}` (length $T$), `gm::Vector{Float64}` (per-day market log growth), `prices::Matrix{Float64}` ($T \times K$ per-ticker prices), and `corpus::MyNewsCorpus` (the scored news corpus the engine will see).

In [3]:
market, gm, prices, corpus = let
    Random.seed!(MASTER_SEED);

    # --- Step 1: Simulate the market index ---
    market = ones(T_DAYS) .* 100.0;
    for t in 2:T_DAYS
        market[t] = market[t - 1] * exp(MARKET_DRIFT + MARKET_VOL * randn());
    end
    gm = log.(market[2:end] ./ market[1:end - 1]);

    # --- Step 2: Generate the synthetic news corpus (zero kappa so news has no own-shock) ---
    scen = build(MyNewsScenario, (
        label = "deep-dive controlled", kappa_pos = 0.0, kappa_neg = 0.0,
        arrival_intensity = 0.4, sentiment_mean = 0.0, sentiment_sd = 0.5));
    corpus = simulate_news_corpus(ones(T_DAYS, K_TICKERS), TICKERS_DEMO, scen; seed = 21);
    score_news_with_claude!(corpus; live = false, cached_noise_sd = 0.10, seed = 22);

    # --- Step 3: Simulate per-ticker prices via SIM with planted ν loadings ---
    prices = ones(T_DAYS, K_TICKERS) .* 100.0;
    for t in 2:T_DAYS, i in 1:K_TICKERS
        g = TRUE_ALPHA[i] +
            TRUE_BETA[i] * gm[t - 1] +
            TRUE_NU[i] * corpus.news_factor[t, i] +
            TRUE_SIGMA_EPS[i] * randn();
        prices[t, i] = prices[t - 1, i] * exp(g);
    end

    println("Generated $(T_DAYS) days of market and per-ticker prices.")
    println("Corpus: $(length(corpus.items)) news items across $(K_TICKERS) tickers.")
    println("Sample item.true_score range: ", round(minimum(it.true_score for it in corpus.items), digits=3),
        " to ", round(maximum(it.true_score for it in corpus.items), digits=3));
    market, gm, prices, corpus
end;

Generated 600 days of market and per-ticker prices.


Corpus: 714 news items across 3 tickers.
Sample item.true_score range: -1.0

 to 1.0


___
## Task 1: Trace One Headline Through the Pipeline
In this task, we follow a single news item from generation through scoring to its appearance in the per-ticker `news_factor` matrix the engine reads. The fields on a [`MyNewsItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyNewsItem) record both the planted `true_score` (the sentiment we asked for) and the `claude_score` (the noisy estimate Claude returned). The cached scoring path uses `cached_noise_sd > 0` to simulate Claude's noise without making a real API call.

> __What should we see?__
>
> The `claude_score` differs from the `true_score` by approximately the cached noise standard deviation; with `cached_noise_sd = 0.10` we expect deviations on the order of $\pm 0.1$. The `news_factor` matrix is the per-day per-ticker mean of the `claude_score` for items mentioning the ticker on that day; days with no items get zero.

The cell-bound result is `traced_item::MyNewsItem` (the specific item we walked) plus a `pretty_table` of the first ten scored items in the corpus.

In [4]:
traced_item = let
    # --- Step 1: Pick the first item with a strongly positive planted score ---
    candidates = filter(it -> it.true_score > 0.3, corpus.items);
    isempty(candidates) && error("No strongly-positive planted scores in this corpus seed.");
    item = candidates[1];

    # --- Step 2: Render the item's fields ---
    println("Traced item:")
    println("  ticker            = ", item.ticker)
    println("  publication_day   = ", item.publication_day)
    println("  source            = ", item.source)
    println("  true_score        = ", round(item.true_score, digits = 3),
        "   (planted by simulate_news_corpus)")
    println("  claude_score      = ", round(item.claude_score, digits = 3),
        "   (returned by score_news_with_claude! with cached_noise_sd = 0.10)")
    println("  noise (Δ)         = ", round(item.claude_score - item.true_score, digits = 3))
    println("  text              = ", item.text[1:min(80, length(item.text))], "...")

    # --- Step 3: Show the news_factor entry the engine reads for this ticker on this day ---
    k = findfirst(==(item.ticker), TICKERS_DEMO);
    factor_value = corpus.news_factor[item.publication_day, k];
    println("\nnews_factor[$(item.publication_day), $(item.ticker)] = ",
        round(factor_value, digits = 3),
        "   (mean of claude_score across items mentioning $(item.ticker) on day $(item.publication_day))")

    # --- Step 4: Tabulate the first ten items in the corpus ---
    rows = NamedTuple[];
    for it in corpus.items[1:min(10, length(corpus.items))]
        push!(rows, (Day = it.publication_day, Ticker = it.ticker,
            True_score = round(it.true_score, digits = 3),
            Claude_score = round(it.claude_score, digits = 3),
            Δ = round(it.claude_score - it.true_score, digits = 3)));
    end
    println("\nFirst 10 items in the corpus:");
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    item
end;

Traced item:
  ticker            = AAA


  publication_day   = 14
  source            = synthetic_news
  true_score        = 

0.488   (planted by simulate_news_corpus)
  claude_score      = 0.391   (returned by score_news_with_claude! with cached_noise_sd = 0.10)
  noise (Δ)         = -0.098
  text              = [stub] AAA news, day 14, planted sentiment 0.488...

news_factor[14, AAA] = 0.488

   (mean of claude_score across items mentioning AAA on day 14)

First 10 items in the corpus:


 ------- -------- ------------ -------------- ---------
    Day   Ticker   True_score   Claude_score         Δ 
  Int64   String      Float64        Float64   Float64 
 ------- -------- ------------ -------------- ---------
      1      AAA        0.062          0.066     0.004
      4      AAA        0.095          0.038    -0.057
      5      AAA         -0.1         -0.118    -0.018
      9      AAA       -0.289         -0.423    -0.134
      9      AAA       -0.199         -0.074     0.125
     11      AAA       -0.274           -0.4    -0.126
     12      AAA       -0.341          -0.28     0.061
     12      AAA        -0.56         -0.603    -0.044
     14      AAA        0.488          0.391    -0.098
     15      AAA       -0.288         -0.515    -0.227
 ------- -------- ------------ -------------- ---------


___
## Task 2: Recover the Planted ν Loadings
In this task, we run [`estimate_sim_with_news`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#estimate_sim_with_news) against the controlled fixture and check whether it recovers the planted [`TRUE_NU`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#TRUE_NU) values. The estimator is a regression of per-ticker log growth on $(g_{m,t}, \text{news\_factor}_{i,t})$; it returns one $\nu_i$ per ticker.

> __What should we see?__
>
> The recovered $\hat\nu_i$ should sit close to the planted $\nu_i$, with the gap shrinking as $T$ grows and as `cached_noise_sd` shrinks. Tickers with larger planted $\nu_i$ are easier to recover (better signal-to-noise in the news factor).

The cell-bound result is `recovery::DataFrame` with one row per ticker recording planted vs recovered values.

In [5]:
recovery = let
    # --- Step 1: Per-ticker per-day log growth from the simulated prices ---
    growth_rates = log.(prices[2:end, :] ./ prices[1:end - 1, :]);
    market_growth = gm;

    # --- Step 2: Run the SIM-with-news estimator ---
    sim_estimated, nu_estimated = estimate_sim_with_news(growth_rates, market_growth,
        corpus.news_factor[2:end, :], TICKERS_DEMO);

    # --- Step 3: Build the recovery scorecard ---
    rows = NamedTuple[];
    for (i, t) in enumerate(TICKERS_DEMO)
        α_hat, β_hat, σε_hat = sim_estimated[t];
        push!(rows, (
            Ticker = t,
            α_true = round(TRUE_ALPHA[i], digits = 5),
            α_hat = round(α_hat, digits = 5),
            β_true = round(TRUE_BETA[i], digits = 3),
            β_hat = round(β_hat, digits = 3),
            ν_true = round(TRUE_NU[i], digits = 4),
            ν_hat = round(nu_estimated[i], digits = 4),
            ν_err_pct = "$(round(100 * (nu_estimated[i] - TRUE_NU[i]) / TRUE_NU[i], digits = 1))%",
        ));
    end
    df = DataFrame(rows);
    println("Planted vs recovered ν loadings (T = $(T_DAYS) days, cached_noise_sd = 0.10):")
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    df
end;

Planted vs recovered ν loadings (T = 600 days, cached_noise_sd = 0.10):


 -------- --------- ---------- --------- --------- --------- --------- -----------
  Ticker    α_true      α_hat    β_true     β_hat    ν_true     ν_hat   ν_err_pct 
  String   Float64    Float64   Float64   Float64   Float64   Float64      String 
 -------- --------- ---------- --------- --------- --------- --------- -----------
     AAA    0.0001    -1.0e-5       1.1     1.089      0.02     0.022       10.1%
     BBB    0.0002    0.00038       0.8       0.8     0.005    0.0068       35.7%
     CCC    5.0e-5   -0.00019       1.4     1.419      0.03    0.0286       -4.8%
 -------- --------- ---------- --------- --------- --------- --------- -----------


___
## Task 3: Held-Out Test for Overfitting
In this task, we split the sample in half, estimate $\nu_i$ on the first half, and check how well the resulting predictions correlate with realized residuals on the held-out second half. If the news factor is a real signal we expect non-trivial out-of-sample correlation; if the in-sample fit is overfitting we expect the out-of-sample correlation to collapse toward zero.

> __What should we see?__
>
> In-sample correlation between the news-factor-implied growth contribution $\hat\nu_i \cdot \text{news\_factor}_{i,t}$ and the realized residual growth $g_i - \hat\alpha_i - \hat\beta_i g_{m,t}$ should be strongly positive. Out-of-sample correlation should also be positive (the planted $\nu_i$ truly drives the price), but smaller than in-sample because of estimator noise. A near-zero out-of-sample number on a real corpus would be a warning that the news signal is being absorbed by the regression's degrees of freedom rather than capturing real structure.

The cell-bound result is `oos_table::DataFrame` reporting in-sample vs out-of-sample correlations per ticker.

In [6]:
oos_table = let
    # --- Step 1: Split point ---
    split = div(T_DAYS, 2);
    growth_rates = log.(prices[2:end, :] ./ prices[1:end - 1, :]);
    market_growth = gm;
    nf = corpus.news_factor[2:end, :];

    # --- Step 2: Estimate on the first half only ---
    sim_train, nu_train = estimate_sim_with_news(
        growth_rates[1:split, :], market_growth[1:split], nf[1:split, :], TICKERS_DEMO);

    # --- Step 3: Per-ticker correlations on each half ---
    rows = NamedTuple[];
    for (i, t) in enumerate(TICKERS_DEMO)
        α_hat, β_hat, σε_hat = sim_train[t];
        ν_hat = nu_train[i];
        # SIM residual growth (price growth minus α + β·gm)
        resid_full = growth_rates[:, i] .- α_hat .- β_hat .* market_growth;
        contrib_full = ν_hat .* nf[:, i];
        cor_in = cor(resid_full[1:split], contrib_full[1:split]);
        cor_out = cor(resid_full[split + 1:end], contrib_full[split + 1:end]);
        push!(rows, (
            Ticker = t,
            ν_true = round(TRUE_NU[i], digits = 4),
            ν_train = round(ν_hat, digits = 4),
            cor_in_sample = round(cor_in, digits = 3),
            cor_oos = round(cor_out, digits = 3),
            shrink = round(cor_out / max(cor_in, 1e-8), digits = 2),
        ));
    end
    df = DataFrame(rows);
    println("In-sample vs out-of-sample correlation (split at day $(split)):")
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    df
end;

In-sample vs out-of-sample correlation (split at day 300):


 -------- --------- --------- --------------- --------- ---------
  Ticker    ν_true   ν_train   cor_in_sample   cor_oos    shrink 
  String   Float64   Float64         Float64   Float64   Float64 
 -------- --------- --------- --------------- --------- ---------
     AAA      0.02    0.0224            0.56     0.546      0.97
     BBB     0.005    0.0072            0.31     0.321      1.04
     CCC      0.03    0.0275           0.529     0.548      1.04
 -------- --------- --------- --------------- --------- ---------


___
## Summary
This optional example traced one headline through the news pipeline, recovered the planted $\nu_i$ loadings via a controlled experiment, and ran a held-out test to assess whether the loadings generalize. The pipeline is what the [Core ProductionEngine notebook](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb) consumes when it joins news severity onto each proposed trade.

> __Key Takeaways:__
>
> * __The Claude scoring step is a noise model:__ Each `MyNewsItem` carries a planted `true_score` and a noisy `claude_score`; the runner reads only the noisy score, just as it would in a live deployment. The cached scoring path lets us reproduce backtests without running real API calls.
> * __ν loadings are recoverable when the experiment is controlled:__ With $T = 600$ days and `cached_noise_sd = 0.10`, the estimator recovers planted $\nu_i$ values within a small percentage error. The recovery degrades when the corpus is small or the noise is large.
> * __Out-of-sample correlation is the honesty test:__ A pipeline that produces large in-sample correlation but near-zero out-of-sample correlation is overfitting; the news factor is being absorbed by the regression rather than capturing structure. We hold out before we trust.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The synthetic corpus is a teaching artifact; production use of LLM-scored news flow demands additional validation, monitoring, and operational controls.

___